# Módulo 3 · Clase 1 — ¿Cuánta gasolina dará este crudo?

**Machine Learning for Petroleum Engineers Using Python** · SLB Ecuador / UDLA

---

Empieza el módulo de machine learning. Este cuaderno va a la par de la
presentación: cada sección lleva el mismo número que la lámina que dice
*"En el cuaderno"*, y cada ejercicio, el mismo número que su lámina.

**El problema de hoy.** A una refinería le ofrecen un crudo. Antes de
comprarlo quiere saber dos cosas, sin esperar los días que toma destilarlo:

1. **¿Cuánta gasolina dará?** La respuesta es un número: es **regresión**.
2. **¿Es agrio y habrá que tratarlo?** La respuesta es una etiqueta: es **clasificación**.

**Cómo se trabajan los ejercicios.** Cada ejercicio tiene una celda vacía.
Pueden escribir el código ustedes o pedírselo al asistente de IA de Colab,
pero siempre **revisen** lo que obtienen: que el código funcione, que el
resultado tenga sentido y que responda la pregunta. Después comparamos con la
solución en la presentación.

**Cómo usar este cuaderno.** Cada celda se ejecuta con ▶ o con
`Shift + Enter`, en orden. Las celdas que no son ejercicios funcionan aunque
no hayan resuelto los ejercicios.

## 0 · Cargar los datos

Entre 1920 y 1983, la Oficina de Minas de Estados Unidos destiló en laboratorio
miles de crudos, casi todos de pozos de ese país. Cada fila de la tabla es
**un crudo**.

| Columna | Qué contiene |
|---|---|
| `origen` | estado de EE. UU. o país de donde salió el crudo |
| `origen_grupo` | el origen, con los que tienen menos de 100 crudos agrupados en "Otros" |
| `roca`, `edad` | roca del yacimiento y su edad geológica |
| `api` | densidad en grados API: más grados, más liviano |
| `tipo` | pesado (menos de 22,3 °API), mediano o liviano (más de 31,1 °API) |
| `azufre_pct` | azufre, en % del peso |
| `agrio` | "sí" si el azufre pasa de 0,5 % |
| `gasolina_pct` | gasolina, en % del barril (**el target de la regresión**) |
| `querosene_pct`, `gasoleo_pct`, `residuo_pct` | los demás productos de la destilación |

La tabla ya viene limpia: sin datos faltantes y con los grupos hechos. Donde no
se registró la roca o la edad dice "Desconocida".

*Fuente: Crude Oil Analysis Database, U.S. Department of Energy (datos abiertos).*

In [ ]:
# pandas: tablas. numpy: cálculos. matplotlib: gráficos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Leer el archivo desde internet
url = "https://raw.githubusercontent.com/cmosquerat/slb-diplomado/main/datos/analisis_crudos.csv"
crudos = pd.read_csv(url)

# Tamaño de la tabla
print(crudos.shape)

# La fila del crudo de Ecuador
crudos.loc[6765]

---
## Parte 2 · Explorar los datos pensando en la tarea

Antes de entrenar cualquier modelo miramos los datos con la tarea en mente:
qué tipos de crudo hay, si la densidad anticipa la gasolina y cuántos crudos
son agrios.

### ✏️ Ejercicio 1

**¿Qué tipos de crudo hay en la tabla?**

1. Cuenten cuántos crudos hay de cada `tipo`.
2. Hagan el histograma de la densidad (`api`).

*Para responder:* ¿qué tipo de crudo domina la tabla? ¿Qué puede significar eso
para un modelo que aprenda de ella?

*Pista:* contar los valores de una columna y hacer un histograma son dos operaciones
del módulo 2.

In [ ]:
# Escriban aquí su código

### ✏️ Ejercicio 2

**¿Cuánta gasolina da cada tipo de crudo?**

1. Calculen el promedio de gasolina de cada `tipo`.
2. Hagan el diagrama de dispersión de `api` contra `gasolina_pct` y calculen la
   correlación de Spearman.

*Para responder:* ¿se cumple lo que esperábamos por la física? ¿Sirve la
densidad para predecir la gasolina?

*Pista:* agrupar por una columna y promediar otra, y la correlación de Spearman, las
vimos en el módulo 2.

In [ ]:
# Escriban aquí su código

### ✏️ Ejercicio 3

**¿Qué crudos son agrios?**

1. Calculen qué porcentaje de los crudos es agrio.
2. Repitan el cálculo para cada `tipo` y para cada `origen_grupo`.

*Para responder:* ¿hay tantos agrios como dulces? ¿El azufre depende de la
densidad? ¿Y del lugar?

*Pista:* una tabla cruzada de dos columnas, con los porcentajes calculados por fila,
responde el segundo punto.

In [ ]:
# Escriban aquí su código

---
## Parte 3 · Codificar: los modelos solo comen números

Por dentro, un modelo hace sumas y multiplicaciones. "Caliza" no se puede
multiplicar: antes de entrenar, cada columna de texto se convierte en números.

| Técnica | Qué hace | Cuándo se usa | En nuestra tabla |
|---|---|---|---|
| **Binarizar** | 2 categorías → 0 y 1 | la columna tiene solo dos valores | `agrio` |
| **Ordinal** | categorías → 0, 1, 2… | las categorías tienen un orden real | `tipo` |
| **One-hot** | una columna de 0 y 1 por categoría | las categorías no tienen orden | `roca`, `edad`, `origen_grupo` |

**La trampa del orden falso:** si numeramos arenisca = 1, caliza = 2,
dolomita = 3, el modelo cree que la caliza está "a mitad de camino" entre las
otras dos. Python no avisa.

### ✏️ Ejercicio 4

**Conviertan la tabla en números.**

1. Binaricen `agrio` en una columna nueva, `agrio_num`: 1 si es "sí", 0 si es "no".
2. Armen la tabla de features `X`: `api`, `azufre_pct` y el one-hot de `roca`,
   `edad` y `origen_grupo`.

*Para responder:* ¿cuántas columnas tiene `X`? ¿Cuántos unos hay en cada fila?

*Pista:* pandas tiene una función para reemplazar valores con un diccionario y otra que
crea las casillas. Ninguna necesita ciclos.

In [ ]:
# Escriban aquí su código

**Punto de control.** La celda siguiente deja listas `agrio_num` y la tabla de
features `X`, que usaremos en el resto de la clase. Ejecútenla aunque hayan
resuelto el ejercicio 4.

In [ ]:
#@title Punto de control: deja listas agrio_num y X
crudos['agrio_num'] = crudos['agrio'].map({'no': 0, 'sí': 1})
X = pd.get_dummies(
    crudos[['api', 'azufre_pct', 'roca', 'edad', 'origen_grupo']],
    columns=['roca', 'edad', 'origen_grupo'],
    dtype=int)
print('X tiene', X.shape[1], 'columnas')

---
## Parte 4 · La mejor recta: regresión lineal

## 4.1 · Nuestro primer modelo con scikit-learn

`scikit-learn` es la librería de machine learning. Siempre son tres pasos:
crear el modelo, entrenarlo con `.fit(X, y)` y predecir con `.predict()`.

El modelo más simple usa una sola feature, la densidad:
**gasolina = b + m × api**. Entrenar es encontrar el intercepto `b` y la
pendiente `m` que dejan los residuos más cortos.

In [ ]:
from sklearn.linear_model import LinearRegression

X_simple = crudos[['api']]     # feature (corchetes dobles: una tabla)
y = crudos['gasolina_pct']     # target

modelo = LinearRegression()
modelo.fit(X_simple, y)        # entrenar
print(modelo.intercept_.round(2), modelo.coef_.round(3))

# Predecir un crudo nuevo de 35 °API
nuevo = pd.DataFrame({'api': [35]})
print(modelo.predict(nuevo).round(1))

### ✏️ Ejercicio 5

**¿Qué tan bueno es nuestro primer modelo?**

1. Calculen el MAE, el RMSE y el R² del modelo de la sección 4.1.
2. Calculen las mismas tres métricas para el "modelo tonto", que siempre predice
   la media de la gasolina.

*Para responder:* ¿le gana la recta al modelo tonto? Expliquen su MAE en una
frase, para un gerente que no conoce la métrica.

*Pista:* las métricas están en `sklearn.metrics`. El RMSE es la raíz del error
cuadrático medio.

In [ ]:
# Escriban aquí su código

## 4.3 · Muchas features: regresión múltiple

Es la misma recta, con varias features. La ecuación tiene **un coeficiente por
cada columna** de `X`, incluidas las casillas del one-hot. El código casi no
cambia: solo cambia la tabla de features.

### ✏️ Ejercicio 6

**¿Ayudan las demás columnas?**

1. Entrenen un modelo con las 29 columnas de `X`.
2. Calculen su MAE, su RMSE y su R².
3. Miren los coeficientes: ¿cuáles son los más grandes?

*Para responder:* ¿cuánto mejoró frente a la recta con la sola densidad? ¿Qué
dicen los coeficientes de California y de Pensilvania?

*Pista:* es el mismo código de la sección 4.1, con otra tabla de features. Los
coeficientes vienen en el mismo orden que las columnas de `X`.

In [ ]:
# Escriban aquí su código

## 4.4 · Estandarizar con `StandardScaler`

Los coeficientes dependen de las unidades: `api` va de 6 a 72 y `azufre_pct`
casi siempre de 0 a 3. Para compararlos hay que poner todas las features en la
misma escala.

**Estandarizar** es calcular el puntaje z de cada columna: a cada valor se le
resta la media y se divide por la desviación estándar. Después, cada feature
tiene media 0 y desviación 1.

In [ ]:
from sklearn.preprocessing import StandardScaler

escalador = StandardScaler()
X_est = escalador.fit_transform(X)

# Volver a una tabla, con los nombres
X_est = pd.DataFrame(X_est, columns=X.columns)
print(X_est[['api', 'azufre_pct']].describe().round(2).loc[['mean', 'std']])

### ✏️ Ejercicio 7

**Ahora sí: ¿cuál variable importa más?**

1. Entrenen un modelo con `X_est`, la tabla estandarizada.
2. Ordenen sus coeficientes por tamaño, sin el signo, y miren los ocho primeros.
3. Calculen su R².

*Para responder:* ¿cuál feature pesa más? ¿Cuántas veces más que la segunda?
¿Cambió el R² al estandarizar?

*Pista:* guarden los coeficientes en una Serie de pandas con los nombres de las columnas:
así se pueden ordenar por su valor absoluto.

In [ ]:
# Escriban aquí su código

### Explicabilidad: la cuenta de un crudo

Un modelo lineal se puede abrir. Cada predicción es una suma: el intercepto más
el aporte de cada feature (su coeficiente por su valor). De las casillas solo
cuentan las que valen 1.

In [ ]:
# Un modelo con las 29 columnas, en sus unidades originales
modelo_todo = LinearRegression().fit(X, y)
coef_todo = pd.Series(modelo_todo.coef_, index=X.columns)

# El crudo de Ecuador: cuánto aporta cada feature a su predicción
fila = X.loc[6765]
aportes = (coef_todo * fila)[fila != 0]
print('intercepto', round(modelo_todo.intercept_, 1))
print(aportes.round(1))
print('predicción', round(modelo_todo.intercept_ + aportes.sum(), 1), '| real', y[6765])

## 4.5 · Separar los datos con `train_test_split`

Hasta ahora medimos el modelo con los mismos datos con los que entrenó. Lo que
importa es cómo predice sobre crudos **que nunca vio**. Por eso se esconde una
parte de los datos antes de entrenar:

- **train** (80 %): el modelo aprende de ellos;
- **test** (20 %): se usan solo para evaluar.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape)

### ✏️ Ejercicio 8

**¿El modelo aprendió o memorizó?**

1. Entrenen el modelo de las 29 columnas solo con `X_train` y `y_train`.
2. Calculen su R² y su MAE en el train.
3. Calculen las dos métricas en el test.

*Para responder:* ¿se parecen las métricas del train y las del test? ¿Qué
significaría que fueran muy distintas?

*Pista:* el modelo se entrena una sola vez, con el train. Después se le piden
predicciones para cada una de las dos partes.

In [ ]:
# Escriban aquí su código

---
## Parte 5 · Clasificación: ¿el crudo es agrio?

## 5.1 · Entrenar: el mismo patrón de scikit-learn

Ahora el target es una etiqueta: `agrio_num` (1 = agrio). El azufre **no** entra
al modelo, porque es la respuesta.

La **regresión logística** hace por dentro la misma suma con pesos de la parte 4,
y la pasa por la curva sigmoide para entregar una **probabilidad** entre 0 y 1.

Cambia el modelo, no el flujo: separar, estandarizar sin fuga (el escalador
aprende solo del train), `.fit()` y predecir.

In [ ]:
from sklearn.linear_model import LogisticRegression

y_c = crudos['agrio_num']              # target: 1 = agrio
X_c = X.drop(columns='azufre_pct')     # el azufre es la respuesta
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42)

escalador = StandardScaler()
Xc_train_est = escalador.fit_transform(Xc_train)  # aprende solo del train
Xc_test_est = escalador.transform(Xc_test)        # al test solo se le aplica

clasificador = LogisticRegression(max_iter=1000)
clasificador.fit(Xc_train_est, yc_train)

# Probabilidad de ser agrio, para cada crudo del test
prob = clasificador.predict_proba(Xc_test_est)[:, 1]
print(prob[:5].round(2))
print('crudos en el test:', len(yc_test), '| agrios:', yc_test.sum())

### ✏️ Ejercicio 9

**¿Qué tan bueno es decir siempre "dulce"?**

1. Para los crudos del test, predigan "dulce" (0) para todos.
2. Calculen el porcentaje de aciertos de esa regla: su **accuracy**.
3. Cuenten cuántos crudos agrios detecta.

*Para responder:* ¿es un buen clasificador? ¿A cuál de los dos detectores de gas
se parece?

*Pista:* la accuracy es el porcentaje de crudos bien clasificados. Está en
`sklearn.metrics`, aunque también se puede calcular comparando dos columnas.

In [ ]:
# Escriban aquí su código

### ✏️ Ejercicio 10

**¿Qué tan bueno es el clasificador con umbral 0,5?**

1. Declaren agrio todo crudo del test con probabilidad de 0,5 o más.
2. Calculen la matriz de confusión.
3. Calculen la accuracy, la precision, el recall y el F1.

*Para responder:* ¿le gana a decir siempre "dulce"? De cada diez crudos agrios,
¿cuántos se le escapan?

*Pista:* comparen `prob` con el umbral para obtener ceros y unos. La matriz y las cuatro
métricas están en `sklearn.metrics`. En la matriz de scikit-learn, las filas son
lo real y las columnas lo predicho, en el orden 0, 1.

In [ ]:
# Escriban aquí su código

### Explicabilidad: ¿por qué el modelo dice "agrio"?

Como estandarizamos, los coeficientes del clasificador se pueden comparar. Uno
positivo sube la probabilidad de agrio; uno negativo la baja.

In [ ]:
coef_c = pd.Series(clasificador.coef_[0], index=X_c.columns)
orden = coef_c.abs().sort_values(ascending=False).index
print(coef_c[orden].head(9).round(2))

### ✏️ Ejercicio 11

**¿Cuánto cuesta cada umbral?**

Supongamos que **tratar un crudo cuesta 4.000 USD** y que **un crudo agrio sin
tratar cuesta 40.000 USD** en corrosión, paradas y multas.

1. Calculen el costo de dos estrategias sin modelo: tratar todos los crudos del
   test y no tratar ninguno.
2. Calculen el costo del clasificador con los umbrales 0,5; 0,3; 0,2; 0,15; 0,1
   y 0,05.

*Para responder:* ¿qué umbral elegirían? ¿Conviene usar el modelo con el
umbral 0,5?

*Pista:* se tratan los crudos que el modelo declara agrios, acierte o no. Los agrios que
se escapan son los falsos negativos. Un ciclo `for` repite la cuenta para cada
umbral.

In [ ]:
# Escriban aquí su código

---
## El flujo que aprendieron hoy

1. **La pregunta:** ¿la respuesta es un número (regresión) o una etiqueta (clasificación)?
2. **Los datos:** el target y las features, solo con lo que se conoce antes.
3. **Explorar:** la física, el desbalance de las etiquetas.
4. **Codificar:** binarizar, ordinal u one-hot, sin inventar un orden.
5. **Separar:** train y test.
6. **Estandarizar:** con el escalador ajustado solo en el train.
7. **Entrenar:** `.fit(X, y)`.
8. **Evaluar y decidir:** MAE, RMSE y R² en regresión; matriz de confusión,
   precision y recall en clasificación; y el umbral, con la matriz de costos.

En los pasos 1, 4 y 8 deciden ustedes. Los demás los puede escribir Python o un
asistente de IA.